# Marketing Measurement Lab — Google Meridian MMM (public fast-reproduction candidate)

**Scope:** 48 fictional US markets, first 120 pre-experiment weeks, Search / Social / Display. All results use **synthetic** data and are exploratory.

**What this demonstrates:** reconstruct the original Meridian model specification; retrieve **publicly versioned inputs and saved Bayesian draws** without your laptop or private Drive; regenerate Google's official results, health and optimization reports; inspect adstock, saturation and rank-normalized convergence diagnostics.

**Important distinction:** This default workflow **restores an existing posterior**; it does not independently refit Meridian. The full refit is an opt-in, compute-intensive experiment provided at the end, not part of the currently verified fast-reproduction claim. It has not received a clean independent test yet.

**Public asset paths (must be uploaded to GitHub before Run all):**
- `mmm_geo_weekly.csv` — public weekly time series at the repository root; original Git blob hash verified.
- `data/mmm/geos.csv` — population for each of the 48 synthetic markets; verified SHA-256.
- `artifacts/mmm/MMM_Meridian_Exploratory.nc` — previously saved posterior, priors, trace and sampling metadata; verified SHA-256.

The GitHub `main` branch is the eventual public default. **During review-branch QA, set `REPO_REF` in Step 2 to the actual branch name.** Then return it to `main` after merging; checksums pin expected exact file contents. If an input changes, this notebook fails instead of silently analyzing a different version. Later, a release can pin a commit SHA and move binary files to a GitHub Release.

**No private Google Drive mount, account login, or file upload is required.** Notebook output is written to `/content/mmm_public_outputs/`. This publication candidate has not yet been freshly run with public URLs; those GitHub files do not currently exist. **Do not label it independently verified until a fresh reader-style run passes after publication.**

**Scientific limitations:** the original 2-chain × 1,000-draw posterior has strict R-hat values as high as 1.3781 for time-baseline terms, 1.1924 for `tau_g`, and 1.0218 for `roi_m`. Meridian's separate health report displays 100/100 under its own criteria. A successful restore and good prediction do **not** prove convergence, identification, or causal validity.

**Boundary:** Last-touch, other MTA, the independent geo-experiment and counterfactual-truth comparison are separate workstreams. Do **not** open `truth_only/` while specifying or evaluating this MMM.

## Step 1 — Install the pinned Meridian version

Run **only this cell first**. Colab resets remove installed packages as well as Python variables. We intentionally use the same version as the original exploratory fit.

In [1]:
%pip install -q "google-meridian==2.2.0"

## Step 2 — Download public, checksummed source data and load the 120-week cohort

This cell downloads three required assets directly from the **public GitHub repository**. It validates SHA-256 for `geos.csv` and the saved posterior, plus the Git blob SHA-1 for the weekly dataset. **No manual upload or Google Drive needed.** The dataset must contain 48 geos × 120 selected weeks = 5,760 geo-weeks.

In [2]:
from pathlib import Path
from importlib.metadata import version
from urllib.request import Request, urlopen
import hashlib
import pandas as pd
from meridian.data import load
from meridian.model import model, spec

REPO = "rahitushar1/marketing-measurement-lab"
REPO_REF = "m3-mmm-public-reproduction"
BASE = f"https://raw.githubusercontent.com/{REPO}/{REPO_REF}"
INPUT_DIR = Path("/content/mmm_public_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA256 = {
    "data/mmm/geos.csv": "b0eba8a59d26995ad9ad7262d5bcbfe6560f607461720634619d42250b4990e9",
    "artifacts/mmm/MMM_Meridian_Exploratory.nc": "c4419cc22c5f0e3a708cbdbdd684a239873793f7fd3c008f407af81d5e67bbd8",
}
EXPECTED_WEEKLY_GIT_BLOB_SHA1 = "7df07a92573810b2c61c43e563102153f55bb14c"

def content_hash(path, algorithm="sha256"):
    digest = hashlib.new(algorithm)
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def git_blob_sha1(path):
    # GitHub's file SHA is sha1('blob <size>\0' + raw bytes).
    raw = path.read_bytes()
    return hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()

def fetch_checked(rel_path, expected_sha256=None, expected_blob_sha1=None):
    target = INPUT_DIR / Path(rel_path).name
    def verified():
        if not target.is_file():
            return False
        if expected_sha256 is not None:
            return content_hash(target) == expected_sha256
        return git_blob_sha1(target) == expected_blob_sha1
    if verified():
        print("CACHED VERIFIED:", rel_path)
        return target
    url = f"{BASE}/{rel_path}"
    request = Request(url, headers={"User-Agent": "MarketingMeasurementLab-reproduction"})
    temporary = target.with_suffix(target.suffix + ".part")
    try:
        with urlopen(request, timeout=120) as response, open(temporary, "wb") as output:
            while block := response.read(1024 * 1024):
                output.write(block)
        temporary.replace(target)
        if not verified():
            target.unlink(missing_ok=True)
            raise ValueError(f"Checksum mismatch: {rel_path}. Do not continue on altered inputs.")
        print("DOWNLOADED AND VERIFIED:", rel_path, f"({target.stat().st_size:,} bytes)")
        return target
    except Exception:
        temporary.unlink(missing_ok=True)
        raise

weekly_path = fetch_checked("mmm_geo_weekly.csv", expected_blob_sha1=EXPECTED_WEEKLY_GIT_BLOB_SHA1)
geo_path = fetch_checked("data/mmm/geos.csv", expected_sha256=EXPECTED_SHA256["data/mmm/geos.csv"])
posterior_path = fetch_checked("artifacts/mmm/MMM_Meridian_Exploratory.nc", expected_sha256=EXPECTED_SHA256["artifacts/mmm/MMM_Meridian_Exploratory.nc"])
print("Meridian version:", version("google-meridian"))

df = pd.read_csv(weekly_path)
geos = pd.read_csv(geo_path)
assert len(geos) == 48 and geos["geo_id"].is_unique
assert set(geos.columns) == {"geo_id", "modeled_population"}
assert geos["modeled_population"].notna().all()
meridian_inputs = df.merge(geos, on="geo_id", how="left", validate="many_to_one")
assert meridian_inputs["modeled_population"].notna().all()
meridian_inputs["revenue_usd"] = meridian_inputs["revenue_cents"] / 100
for ch in ["search", "social", "display"]:
    meridian_inputs[f"{ch}_spend_usd"] = meridian_inputs[f"{ch}_spend_cents"] / 100
weeks = sorted(meridian_inputs["week_start"].unique())[:120]
meridian_train = meridian_inputs.loc[meridian_inputs["week_start"].isin(weeks)].copy()
assert len(weeks) == 120
assert meridian_train["geo_id"].nunique() == 48
assert len(meridian_train) == 5_760
assert not meridian_train.duplicated(["geo_id", "week_start"]).any()
print("READY: markets=48, weeks=120, geo-weeks=5,760")
print("Training range:", weeks[0], "through", weeks[-1])


DOWNLOADED AND VERIFIED: mmm_geo_weekly.csv (752,881 bytes)
DOWNLOADED AND VERIFIED: data/mmm/geos.csv (352 bytes)
DOWNLOADED AND VERIFIED: artifacts/mmm/MMM_Meridian_Exploratory.nc (12,576,270 bytes)
Meridian version: 2.2.0
READY: markets=48, weeks=120, geo-weeks=5,760
Training range: 2023-01-02 through 2025-04-14


## Step 3 — Rebuild the **same** Meridian model specification

All three channels use **impressions** as media inputs; dollar spend is supplied separately. Four controls model demand and business conditions. This recreates the original balanced 20% holdout, seeded with 62191. No fitting happens in this cell.

In [3]:
media_channels = {
    "search_impressions": "search",
    "social_impressions": "social",
    "display_impressions": "display",
}
spend_channels = {
    "search_spend_usd": "search",
    "social_spend_usd": "social",
    "display_spend_usd": "display",
}

columns = load.CoordToColumns(
    time="week_start", geo="geo_id", kpi="revenue_usd",
    population="modeled_population",
    controls=[
        "organic_queries", "competitor_index_bps",
        "promotion_discount_bps", "list_price_index_bps",
    ],
    media=list(media_channels),
    media_spend=list(spend_channels),
)
loader = load.DataFrameDataLoader(
    df=meridian_train, coord_to_columns=columns,
    kpi_type="revenue", media_to_channel=media_channels,
    media_spend_to_channel=spend_channels, currency_code="USD",
)
meridian_data = loader.load()
model_spec = spec.ModelSpec(
    holdout=spec.HoldoutSpec(
        spec=spec.RandomHoldoutSpec(ratio=0.20, seed=62191, balanced=True)
    )
)
mmm = model.Meridian(input_data=meridian_data, model_spec=model_spec)
assert meridian_data.kpi.shape == (48, 120)
print("MODEL CREATED (not fitted)")
print("Outcome:", meridian_data.kpi.shape)
print("Media:", meridian_data.media.shape)

/usr/local/lib/python3.13/dist-packages/meridian/data/input_data.py:612: UserWarning: Revenue from the `kpi` data is used when `kpi_type`=`revenue`. `revenue_per_kpi` is ignored.
  warnings.warn(


MODEL CREATED (not fitted)
Outcome: (48, 120)
Media: (48, 120, 3)


## Step 4 — Create a local report output folder

Unlike the original working notebook, reports are stored in this Colab session's temporary `/content/` directory. Download them before closing the runtime if you want to keep local copies. This does not require personal Google Drive permissions.

In [4]:
from pathlib import Path

output_dir = Path("/content/mmm_public_outputs")
output_dir.mkdir(parents=True, exist_ok=True)
assert posterior_path.is_file(), "Run Step 2 to download the public posterior first."
print("Output directory:", output_dir)
print("Saved posterior available:", posterior_path, posterior_path.stat().st_size, "bytes")

Output directory: /content/mmm_public_outputs
Saved posterior available: /content/mmm_public_inputs/MMM_Meridian_Exploratory.nc 12576270 bytes


## Step 5 — Restore the checksummed **public posterior** (sampling skipped)

This posterior is the stored result of a previous Bayesian fit. Restoring it lets any reader regenerate charts and optimization estimates cheaply. **This is not an independent refit.** If it is absent or differs from the published checksum, the workflow stops rather than silently retraining.

In [5]:
import arviz as az

assert posterior_path.is_file(), "Download public posterior using Step 2 first."
saved_draws = az.from_netcdf(posterior_path)
assert "posterior" in saved_draws.groups(), "Posterior group missing."
assert "roi_m" in saved_draws.posterior, "ROI draws missing."
mmm = model.Meridian(
    input_data=meridian_data,
    model_spec=model_spec,
    inference_data=saved_draws,
)
roi = mmm.inference_data.posterior["roi_m"]
assert roi.sizes["chain"] == 2 and roi.sizes["draw"] == 1000
assert roi.sizes["media_channel"] == 3
print("PUBLIC POSTERIOR RESTORED. Sampling skipped.")
print("ROI medians:", roi.median(dim=("chain", "draw")).values.round(2))
print("Posterior SHA-256:", content_hash(posterior_path))

/usr/local/lib/python3.13/dist-packages/arviz/data/inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(


PUBLIC POSTERIOR RESTORED. Sampling skipped.
ROI medians: [3.2  6.32 2.12]
Posterior SHA-256: c4419cc22c5f0e3a708cbdbdd684a239873793f7fd3c008f407af81d5e67bbd8


## Step 6 — Produce Google's **official model-results HTML report**

This does not refit the model. The two-page report includes model fit, revenue/channel contribution, performance breakdown, and response curves. Some visualizations (adstock and Hill saturation) require separate plotting later; this report is the starting point, **not** the entire case study.

If generation fails because of convergence or other model constraints, keep the error for diagnosis—we should not hide it or represent the report as complete.

In [6]:
from meridian.analysis import summarizer
from IPython.display import display, HTML

report_path = output_dir / "MMM_Meridian_Model_Results.html"
summarizer.Summarizer(mmm).output_model_results_summary(
    filename=report_path.name,
    filepath=str(output_dir),
    start_date=str(weeks[0])[:10],
    end_date=str(weeks[-1])[:10],
)
assert report_path.is_file() and report_path.stat().st_size > 0
print("HTML REPORT CREATED:", report_path)
print("Size KB:", round(report_path.stat().st_size / 1024, 1))
display(HTML(filename=str(report_path)))

/usr/local/lib/python3.13/dist-packages/meridian/analysis/analyzer.py:3082: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/meridian/analysis/analyzer.py:560: UserWarning: Setting `use_kpi=True` has no effect when `kpi_type=REVENUE` since in this case, KPI is equal to revenue.
  warnings.warn(


HTML REPORT CREATED: /content/mmm_public_outputs/MMM_Meridian_Model_Results.html
Size KB: 426.1


Dataset,R-squared,MAPE,wMAPE
Training Data,1.00,0.8%,0.8%
Testing Data,0.88,6%,6%
All Data,0.99,1%,1%


## Step 7 — Review official model health

These are the library's built-in screening checks, **not proof of causal accuracy**. Our separate strict convergence diagnostics should be reported alongside this card.


In [7]:
from meridian.analysis.review import reviewer

health_summary = reviewer.ModelReviewer(mmm).run()
health_summary.output_model_health_card(
    filename="MMM_Meridian_Model_Health.html",
    filepath=str(output_dir),
)
health_path = output_dir / "MMM_Meridian_Model_Health.html"
assert health_path.is_file() and health_path.stat().st_size > 0
print("MODEL HEALTH CARD SAVED:", health_path)

/tmp/ipykernel_1858/3637403134.py:3: DeprecationWarning: The `meridian` argument is deprecated. Please use `model_context` and `inference_data` instead.
  health_summary = reviewer.ModelReviewer(mmm).run()
/usr/local/lib/python3.13/dist-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(


MODEL HEALTH CARD SAVED: /content/mmm_public_outputs/MMM_Meridian_Model_Health.html


## Step 8 — Stricter Bayesian convergence review

**Why:** Meridian's model-health card passed convergence under a check described as R-hat < 1.20, yet a separate rank-normalized calculation finds substantially higher values in some latent baseline parameters. These may use different variable sets or calculations. Do not assume either check proves causal accuracy.

In [8]:
import arviz as az
import numpy as np

rhat_ds = az.rhat(mmm.inference_data.posterior, method="rank")
summary = []
all_rhats = []
for name in rhat_ds.data_vars:
    values = np.asarray(rhat_ds[name].values, dtype=float).ravel()
    finite = values[np.isfinite(values)]
    if finite.size:
        summary.append((name, float(finite.max())))
        all_rhats.append(finite)
summary.sort(key=lambda item: item[1], reverse=True)
print("Highest R-hat by parameter group:")
for parameter, maximum in summary[:12]:
    print(f"{parameter:25s} {maximum:.4f}")
combined = np.concatenate(all_rhats)
print("Elements above 1.01:", f"{100 * np.mean(combined > 1.01):.1f}%")
print("Elements above 1.05:", f"{100 * np.mean(combined > 1.05):.1f}%")
print("NOTE: Non-finite R-hat elements are excluded from these percentages.")

/usr/local/lib/python3.13/dist-packages/arviz/stats/diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)


Highest R-hat by parameter group:
knot_values               1.3781
mu_t                      1.3781
tau_g                     1.1924
gamma_gc                  1.1233
beta_gm                   1.1031
eta_m                     1.0596
beta_m                    1.0332
roi_m                     1.0218
gamma_c                   1.0054
xi_c                      1.0047
alpha_m                   1.0022
ec_m                      1.0020
Elements above 1.01: 52.2%
Elements above 1.05: 44.7%
NOTE: Non-finite R-hat elements are excluded from these percentages.


## Step 9 — Mechanisms: adstock and saturation

**What:** Adstock describes carryover after exposure; Hill curves describe diminishing response to increasing media exposure. Neither curve by itself is an ROI estimate or evidence of causality. The gray media-distribution marks help identify poorly supported extrapolation.

In [9]:
from meridian.analysis import visualizer
from IPython.display import display

media_effects = visualizer.MediaEffects(mmm)
display(media_effects.plot_adstock_decay())
for channel, chart in media_effects.plot_hill_curves().items():
    print("Hill saturation —", channel)
    display(chart)

alt.FacetChart(...)

Hill saturation — media


alt.FacetChart(...)

## Step 10 — Historical fixed-budget optimization (explicit bounds)

**Question:** What change in the **same historical total budget** maximizes *modeled* incremental revenue? Each channel is constrained to **70%–130% of historical spend**. The *first* optimization run relied on Meridian's implicit default ±30% setting. Here the same choice is made **explicit and auditable**; it was not a business rule provided by the user.

**Interpretation:** An allocation at an upper/lower bound is constrained by these rules and is not automatically a global optimum. The estimates remain exploratory because of unresolved posterior convergence and confounding/causal validation concerns.

In [10]:
from meridian.analysis import optimizer

budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize(
    fixed_budget=True,
    use_posterior=True,
    spend_constraint_lower=0.30,
    spend_constraint_upper=0.30,
)
optimization_results.output_optimization_summary(
    "MMM_Meridian_Budget_Optimization.html",
    str(output_dir),
)
opt_path = output_dir / "MMM_Meridian_Budget_Optimization.html"
assert opt_path.is_file() and opt_path.stat().st_size > 0
print("EXPLORATORY OPTIMIZATION REPORT:", opt_path)

EXPLORATORY OPTIMIZATION REPORT: /content/mmm_public_outputs/MMM_Meridian_Budget_Optimization.html


## Step 11 — Interpretation, limitations, and publication gate

**Observed from the reports supplied by the user (do not mistake this text for a fresh execution):**

| Measure | Reported result |
|---|---|
| Train / test R² | 0.9978 / 0.8803 (national report aggregate) |
| Test wMAPE | 6.14% nationally; geo-level errors can be higher |
| Media revenue contribution | Social ≈ $1.065M, Search ≈ $612K, Display ≈ $147K, baseline ≈ $6.49M |
| Posterior *mean* channel ROI | Search 3.21x, Social 6.43x, Display 3.64x |
| Posterior *median* channel ROI | Search ≈ 3.20x, Social ≈ 6.32x, Display ≈ 2.12x |
| Baseline annualized? | **No** — all dollar figures are over the full modeled historical period |
| Fixed historical budget | ≈ $396.9K |
| Exploratory allocation | Social +$49.7K, Search -$43.1K, Display -$6.6K |
| Modeled incremental revenue lift | ≈ $68,228 (+3.74%) |
| Constraint | ±30% of each channel's historical spend; Social reaches +30% |

**Open issues before decision-grade use:**

1. Rank-normalized R-hat max ≈ 1.378 for baseline terms, ≈ 1.192 for geographic variation and ≈ 1.022 for `roi_m`. Verify effective sample sizes and diagnose mixing with a focused follow-on fit if warranted.
2. Large model fit scores do not establish correct causal attribution, especially when latent consumer intent affects both media and purchases.
3. Channel ROI credible intervals are material, especially Display. The optimizer has uncertainty and depends on historical cost/response assumptions.
4. After publication, verify a fresh Colab run from the public assets, with no private files, and record the output and exact commit.
5. **Do not inspect `truth_only/` while developing estimates.** Evaluate the MMM, MTA, and experiment against synthetic truth during the later triangulation milestone.

**Publication gate:** Commit public inputs and posterior only with explicit owner authorization, then run this notebook freshly with no private Drive access. Document the commit, pin `REPO_REF` to that immutable commit, and publish the exact outputs. Keep model-accuracy, convergence, and causal-truth limits visible. This notebook is *fast restoration*, not proof of independent refitting.

### Optional full refit — compute-intensive and **not part of default fast reproduction**

This is a separate research pathway. Set `RUN_FULL_REFIT = True` **deliberately** to sample new priors and posterior from the public model configuration. It is expected to use substantial compute. The full-refit pathway is **not yet verified in a fresh Colab session**, and a fresh fit may reproduce the convergence problems rather than fix them. It must be separately diagnosed before calling the model fully reproduced.

In [11]:
RUN_FULL_REFIT = False  # Default fast reproduction deliberately skips Bayesian sampling.
if RUN_FULL_REFIT:
    from meridian.model import model
    fresh_mmm = model.Meridian(input_data=meridian_data, model_spec=model_spec)
    fresh_mmm.sample_prior(500, seed=62191)
    fresh_mmm.sample_posterior(
        n_chains=2, n_adapt=1000, n_burnin=500, n_keep=1000, seed=62193
    )
    fresh_path = output_dir / "MMM_Meridian_New_Full_Refit_Unreviewed.nc"
    fresh_mmm.inference_data.to_netcdf(fresh_path)
    checked = az.from_netcdf(fresh_path)
    assert "posterior" in checked.groups()
    print("NEW, UNREVIEWED POSTERIOR SAVED:", fresh_path)
    print("New posterior ROI medians:", checked.posterior["roi_m"].median(dim=("chain", "draw")).values)
    print("Review R-hat and other diagnostics before using this refit for decisions.")
else:
    print("Full refit SKIPPED. Original saved posterior used for fast reproduction.")

Full refit SKIPPED. Original saved posterior used for fast reproduction.


## Step 12 — Verify and download the generated reports

The first run should show `PASS` for each of the three HTML reports. Reports are saved locally to Colab's temporary output folder. This is a **file-existence check**, not a check of scientific validity or identical numerical results.

In [12]:
from pathlib import Path

reports = [
    "MMM_Meridian_Model_Results.html",
    "MMM_Meridian_Model_Health.html",
    "MMM_Meridian_Budget_Optimization.html",
]
for name in reports:
    path = output_dir / name
    print(f"{'PASS' if path.is_file() and path.stat().st_size > 0 else 'MISSING'}: {name}")
assert all((output_dir / name).is_file() and (output_dir / name).stat().st_size > 0 for name in reports)
print("ALL THREE REPORTS VERIFIED (files exist; check their contents before interpreting).")

# Optional: download each HTML report to your computer from the Colab file browser
# (Files sidebar > /content/mmm_public_outputs).

PASS: MMM_Meridian_Model_Results.html
PASS: MMM_Meridian_Model_Health.html
PASS: MMM_Meridian_Budget_Optimization.html
ALL THREE REPORTS VERIFIED (files exist; check their contents before interpreting).
